# [Optional] Practice with LEFT JOINs — Extended

Use these code exercises to practice working with LEFT JOINs! These code blocks are optional, and you may choose to spend more or less time with them depending on your level of familiarity with SQL.

This extended version includes the original two exercises plus additional challenges that involve more tables from the Chinook schema (customers, invoices, artists, albums, playlists, genres, employees).

**All code cells below contain the full working SQL solutions with comments.** You can run them directly or clear the cell first if you want to solve the exercise yourself.

## Data Schema

When working with joins, you should reference the data schema to understand the relationships between tables.

The data schema available on Code Blocks is slightly different from the one you saw on the Demos. All the instructions and the following image contain the correct schema for these exercises.

![Database Schema](data-schema.png)

### Key relationships (reminder)

| From table       | To table         | Join key(s)                  |
|------------------|------------------|------------------------------|
| tracks           | invoice_items    | TrackId                      |
| tracks           | albums           | AlbumId                      |
| albums           | artists          | ArtistId                     |
| tracks           | genres           | GenreId                      |
| tracks           | media_types      | MediaTypeId                  |
| playlist_track   | tracks           | TrackId                      |
| playlist_track   | playlists        | PlaylistId                   |
| customers        | invoices         | CustomerId                   |
| invoices         | invoice_items    | InvoiceId                    |
| employees        | employees        | ReportsTo (self-join)        |
| employees        | customers        | SupportRepId                 |

---
## **Exercise 1: Unsold tracks**

The sales department wants to analyze the sales for each track. Identifying high sales or tracks that have never been sold will help them create promotional campaigns.

### Instructions

Write a SQLite query that:

- Selects columns `TrackId` and `Name` from the `tracks` table, and `InvoiceId` from the `invoice_items` table. Make sure to get **all** tracks in the `tracks` table.

In [ ]:
-- Exercise 1: Return every track together with any invoice it appears on.
-- LEFT JOIN keeps all rows from tracks even when there is no matching sale.
SELECT
    t.TrackId,          -- unique identifier of the track
    t.Name,             -- track title
    ii.InvoiceId        -- will be NULL when the track was never sold
FROM tracks t
LEFT JOIN invoice_items ii
    ON t.TrackId = ii.TrackId;   -- match on the common key

**Expected output (first 25 rows):**

```
+---------+-----------------------------------------+-----------+
| TrackId | Name                                    | InvoiceId |
+---------+-----------------------------------------+-----------+
| 1       | For Those About To Rock (We Salute You)| 108       |
| 2       | Balls to the Wall                       | 1         |
| 2       | Balls to the Wall                       | 214       |
| 3       | Fast As a Shark                         | 319       |
| 4       | Restless and Wild                       | 1         |
| 5       | Princess of the Dawn                    | 108       |
| 6       | Put The Finger On You                   | 2         |
| 7       | Let's Get It Up                         | None      |
| 8       | Inject The Venom                        | 2         |
| 8       | Inject The Venom                        | 214       |
| 9       | Snowballed                              | 108       |
| 9       | Snowballed                              | 319       |
| 10      | Evil Walks                              | 2         |
| 11      | C.O.D.                                  | None      |
| 12      | Breaking The Rules                      | 2         |
| 13      | Night Of The Long Knives                | 108       |
| 14      | Spellbound                              | 214       |
| 15      | Go Down                                 | 319       |
| 16      | Dog Eat Dog                             | 3         |
| 17      | Let There Be Rock                       | None      |
| 18      | Bad Boy Boogie                          | None      |
| 19      | Problem Child                           | 109       |
| 20      | Overdose                                | 3         |
| 20      | Overdose                                | 214       |
| 21      | Hell Ain't A Bad Place To Be            | 319       |
+---------+-----------------------------------------+-----------+
(Output limit exceeded, 25 of 3759 total rows shown)
```

**Hints:**

- Use the `SELECT` statement to choose the columns you need from both tables.
- Join the `tracks` table with the `invoice_items` table using the `TrackId` column.
- A `LEFT JOIN` will include all tracks, even those that have not been sold (i.e., without corresponding entries in invoice_items).

---
## **Exercise 2: Tracks that have never been sold**

To decide which tracks might need promotional efforts or removal from the catalog, you want a list of tracks that have never been sold. You can use the previous query as a starter.

### Instructions

Write a SQLite query that:

- Returns `TrackId` and `Name` from the `tracks` table of tracks that have never been sold.
- You will need to look for `NULL` values in `InvoiceId` of the joined table.

In [ ]:
-- Exercise 2: Keep only the tracks that have never been sold.
-- Pattern: LEFT JOIN + WHERE right-side key IS NULL  (classic "anti-join")
SELECT
    t.TrackId,
    t.Name
FROM tracks t
LEFT JOIN invoice_items ii
    ON t.TrackId = ii.TrackId
WHERE ii.InvoiceId IS NULL;   -- no matching invoice_item row exists

**Expected output (first 25 rows):**

```
+---------+---------------------------------------+
| TrackId | Name                                  |
+---------+---------------------------------------+
| 7       | Let's Get It Up                       |
| 11      | C.O.D.                                |
| 17      | Let There Be Rock                     |
| 18      | Bad Boy Boogie                        |
| 22      | Whole Lotta Rosie                     |
| 23      | Walk On Water                         |
| 27      | Dude (Looks Like A Lady)              |
| 29      | Cryin'                                |
| 33      | The Other Side                        |
| 34      | Crazy                                 |
| 35      | Eat The Rich                          |
| 40      | Perfect                               |
| 41      | Hand In My Pocket                     |
| 45      | Head Over Feet                        |
| 46      | Mary Jane                             |
| 47      | Ironic                                |
| 50      | You Oughta Know (Alternate)           |
| 51      | We Die Young                          |
| 52      | Man In The Box                        |
| 56      | Love, Hate, Love                      |
| 58      | Sunshine                              |
| 59      | Put You Down                          |
| 63      | Desafinado                            |
| 64      | Garota De Ipanema                     |
| 65      | Samba De Uma Nota Só (One Note Samba)|
+---------+---------------------------------------+
(Output limit exceeded, 25 of 1519 total rows shown)
```

**Hints:**

- Copy the query from Exercise 1 (remove the selection of `InvoiceId`).
- Filter for `NULL` values in the `InvoiceId` column to identify tracks that were never purchased.

---
## **Exercise 3: Customers who never purchased anything**

Marketing wants a list of customers who registered but never made a purchase. These customers might be good targets for a first-purchase promotion.

### Instructions

Write a SQLite query that:

- Returns `CustomerId`, `FirstName`, `LastName`, and `Email` from the `customers` table.
- Includes only customers who have **no** matching rows in the `invoices` table.
- Order the results by `LastName`, then `FirstName`.

In [ ]:
-- Exercise 3: Find customers with zero invoices.
-- Same anti-join pattern as Exercise 2, applied to customers ↔ invoices.
SELECT
    c.CustomerId,
    c.FirstName,
    c.LastName,
    c.Email
FROM customers c
LEFT JOIN invoices i
    ON c.CustomerId = i.CustomerId
WHERE i.InvoiceId IS NULL          -- customer has no invoices
ORDER BY c.LastName, c.FirstName;

**Note:** In the standard Chinook database every customer has at least one invoice, so this query typically returns **0 rows**. That is expected and still a valid practice of the LEFT JOIN + IS NULL pattern.

**Hints:**

- Start from `customers` and LEFT JOIN to `invoices` on `CustomerId`.
- Filter where `invoices.InvoiceId IS NULL`.
- Use `ORDER BY` for sorting.

---
## **Exercise 4: Artists with no albums**

The catalog team wants to clean up the database. Find any artists that exist but have never released an album (according to the data).

### Instructions

Write a SQLite query that:

- Returns `ArtistId` and `Name` from the `artists` table.
- Includes only artists that have **no** corresponding rows in the `albums` table.
- Order by artist name.

In [ ]:
-- Exercise 4: Artists that have never released an album.
-- Anti-join: artists LEFT JOIN albums, keep rows where AlbumId is NULL.
SELECT
    a.ArtistId,
    a.Name
FROM artists a
LEFT JOIN albums al
    ON a.ArtistId = al.ArtistId
WHERE al.AlbumId IS NULL           -- no album belongs to this artist
ORDER BY a.Name;

**Hints:**

- LEFT JOIN `artists` to `albums` on `ArtistId`.
- Keep only rows where `albums.AlbumId IS NULL`.

---
## **Exercise 5: Tracks not present in any playlist**

Some tracks may exist in the catalog but are never added to any playlist. Identify them.

### Instructions

Write a SQLite query that:

- Returns `TrackId` and `Name` from the `tracks` table.
- Includes only tracks that do **not** appear in the `playlist_track` table.
- Order by `TrackId`.

In [ ]:
-- Exercise 5: Tracks that never appear in any playlist.
-- playlist_track is the bridge table; absence of a row means "not in any playlist".
SELECT
    t.TrackId,
    t.Name
FROM tracks t
LEFT JOIN playlist_track pt
    ON t.TrackId = pt.TrackId
WHERE pt.PlaylistId IS NULL        -- no playlist contains this track
ORDER BY t.TrackId;

**Hints:**

- LEFT JOIN `tracks` to `playlist_track` on `TrackId`.
- Filter where `playlist_track.PlaylistId IS NULL` (or any column from the right table).

---
## **Exercise 6: Genres with no tracks**

Find any genres that are defined but currently have zero tracks assigned to them.

### Instructions

Write a SQLite query that:

- Returns `GenreId` and `Name` from the `genres` table.
- Includes only genres that have no matching tracks.
- Order by genre name.

In [ ]:
-- Exercise 6: Genres that currently have zero tracks.
-- Useful for data-quality checks (orphaned lookup values).
SELECT
    g.GenreId,
    g.Name
FROM genres g
LEFT JOIN tracks t
    ON g.GenreId = t.GenreId
WHERE t.TrackId IS NULL            -- no track belongs to this genre
ORDER BY g.Name;

**Hints:**

- LEFT JOIN `genres` to `tracks` on `GenreId`.
- Keep rows where `tracks.TrackId IS NULL`.

---
## **Exercise 7: Number of sales per track (including zero)**

Instead of only listing unsold tracks, produce a complete list of every track together with how many times it was sold (including tracks sold zero times).

### Instructions

Write a SQLite query that:

- Returns `TrackId`, `Name`, and a column named `TimesSold` that counts the number of times the track appears in `invoice_items`.
- Includes **all** tracks (even those never sold → `TimesSold = 0`).
- Order by `TimesSold` descending, then by `Name`.

In [ ]:
-- Exercise 7: Count how many times each track was sold (including zeros).
-- COUNT on a right-side column returns 0 for unmatched left-side rows.
SELECT
    t.TrackId,
    t.Name,
    COUNT(ii.InvoiceLineId) AS TimesSold   -- 0 when no sales exist
FROM tracks t
LEFT JOIN invoice_items ii
    ON t.TrackId = ii.TrackId
GROUP BY t.TrackId, t.Name                 -- one row per track
ORDER BY TimesSold DESC, t.Name;

**Hints:**

- LEFT JOIN `tracks` to `invoice_items`.
- Use `COUNT(ii.InvoiceLineId)` or `COUNT(ii.TrackId)` — counting a column from the right side correctly yields 0 for unmatched rows.
- Group by the track columns.

---
## **Exercise 8: Employees and their direct reports (self-join)**

Using a self-join on the `employees` table, list every employee together with the name of the person they report to (if any).

### Instructions

Write a SQLite query that:

- Returns:
  - `EmployeeId`
  - Employee full name (`FirstName || ' ' || LastName`) as `EmployeeName`
  - Manager full name as `ManagerName` (NULL if the employee has no manager)
- Uses a LEFT JOIN of `employees` to itself on `ReportsTo`.
- Order by `EmployeeName`.

In [ ]:
-- Exercise 8: Self-join to show each employee and their manager.
-- LEFT JOIN is required so the top-level manager (ReportsTo IS NULL) still appears.
SELECT
    e.EmployeeId,
    e.FirstName || ' ' || e.LastName AS EmployeeName,
    m.FirstName || ' ' || m.LastName AS ManagerName   -- NULL for the top boss
FROM employees e
LEFT JOIN employees m
    ON e.ReportsTo = m.EmployeeId                     -- self-join on ReportsTo
ORDER BY EmployeeName;

**Hints:**

- Alias the employees table twice (e.g., `e` for employee and `m` for manager).
- Join condition: `e.ReportsTo = m.EmployeeId`.
- Because it is a LEFT JOIN, the top-level employee (who reports to no one) will have `NULL` for the manager columns.

---
## **Exercise 9: Albums with no tracks sold**

Find albums where **none** of their tracks have ever been sold. (This requires joining three tables.)

### Instructions

Write a SQLite query that:

- Returns `AlbumId` and `Title` from the `albums` table.
- Includes only albums for which **no** track has a matching row in `invoice_items`.
- Order by album title.

In [ ]:
-- Exercise 9: Albums whose tracks have never been sold.
-- Multi-table LEFT JOIN + GROUP BY + HAVING COUNT = 0.
SELECT
    al.AlbumId,
    al.Title
FROM albums al
LEFT JOIN tracks t
    ON al.AlbumId = t.AlbumId              -- get every track of the album
LEFT JOIN invoice_items ii
    ON t.TrackId = ii.TrackId              -- check if any of those tracks sold
GROUP BY al.AlbumId, al.Title
HAVING COUNT(ii.InvoiceLineId) = 0         -- zero sales across the whole album
ORDER BY al.Title;

**Hints:**

- You will need `albums` → `tracks` → `invoice_items`.
- One clean approach is to LEFT JOIN tracks and then LEFT JOIN invoice_items, then group by album and keep only albums where `COUNT(ii.InvoiceLineId) = 0`.
- Alternatively you can use a correlated subquery or `NOT EXISTS`, but try to stay with LEFT JOINs for this exercise.

---
## **Exercise 10: Playlists with zero tracks**

Identify playlists that currently contain no tracks.

### Instructions

Write a SQLite query that:

- Returns `PlaylistId` and `Name` from the `playlists` table.
- Includes only playlists that have no rows in `playlist_track`.
- Order by playlist name.

In [ ]:
-- Exercise 10: Empty playlists (no tracks assigned).
-- Anti-join against the bridge table playlist_track.
SELECT
    p.PlaylistId,
    p.Name
FROM playlists p
LEFT JOIN playlist_track pt
    ON p.PlaylistId = pt.PlaylistId
WHERE pt.TrackId IS NULL               -- playlist has no entries
ORDER BY p.Name;

**Hints:**

- LEFT JOIN `playlists` to `playlist_track` on `PlaylistId`.
- Filter where `playlist_track.TrackId IS NULL`.

---
## Summary of patterns practiced

| Pattern | Typical use |
|---------|-------------|
| `LEFT JOIN` + `IS NULL` | Find records in the left table that have **no match** on the right ("anti-join") |
| `LEFT JOIN` + `COUNT(right_column)` | Count related rows while still including left-side rows that have zero matches |
| Self `LEFT JOIN` | Hierarchy / reporting relationships (employees → managers) |
| Multiple LEFT JOINs + `GROUP BY` + `HAVING COUNT(...) = 0` | Multi-table "no related activity" questions |

Keep practicing by inventing your own questions on the remaining tables (media_types, invoice details, support representatives, etc.). The same LEFT JOIN + IS NULL / COUNT pattern will solve most of them.